::: {.callout-important}
## Main Idea
Computing the gradient of a loss (whether through a discretized glacier solver or through a neural network) is the same computation: the *adjoint method*. Reverse-mode automatic differentiation ("backprop") is the adjoint method, generated from your forward code.
:::

- Glaciology has used adjoint models for inversions since MacAyeal (1993).
- Modern scientific ML calls the same computation backpropagation.
- We derive the adjoint equation, see why it beats the obvious approach, and show backprop is the same principle.

In [1]:
import jax, jax.numpy as jnp, numpy as np, diffrax as dfx
jax.config.update("jax_enable_x64", True)
print("jax", jax.__version__, "| diffrax", dfx.__version__)

jax 0.10.2 | diffrax 0.7.2


## 1 · The problem: a gradient through a constraint

A scalar loss $L(u,\theta)\in\mathbb{R}$ depends on parameters $\theta\in\mathbb{R}^p$ through a state $u\in\mathbb{R}^n$ — but $u$ is not given explicitly. It is defined implicitly by a constraint:

$$g(u,\theta) = 0,\qquad g:\mathbb{R}^n\times\mathbb{R}^p\to\mathbb{R}^n \quad\Longrightarrow\quad u = u(\theta).$$

Same shape across very different problems:

- *Glacier inversion*: $g=0$ could be the discretized ice-flow equation; $\theta$ parameterizes $A(T)$ (the lecture's §4).
- *Steady state / implicit solve*: $g=0$ is the fixed-point or root condition.
- *Neural network* (the payoff of §3): $g=0$ is the stack of layer equations.

Goal: $\dfrac{dL}{d\theta}\in\mathbb{R}^p$, computed *without* a cost that scales with the number of parameters $p$.

## 2 · Two ways to get the gradient

We want $dL/d\theta$, but $u$ is tied to $\theta$ by the constraint. Three steps:

**Step 1: chain rule.** $L$ depends on $\theta$ directly and through $u(\theta)$:

$$\frac{dL}{d\theta} = \frac{\partial L}{\partial\theta} + \frac{\partial L}{\partial u}\,\frac{du}{d\theta},
\qquad \frac{\partial L}{\partial u}\in\mathbb{R}^{1\times n},\quad \frac{du}{d\theta}\in\mathbb{R}^{n\times p}.$$

The hard part is $du/d\theta$ — how the state responds to the parameters.

**Step 2: differentiate the constraint.** Since $g(u(\theta),\theta)=0$ for all $\theta$:

$$\frac{\partial g}{\partial u}\frac{du}{d\theta} + \frac{\partial g}{\partial\theta} = 0
\quad\Longrightarrow\quad
\frac{du}{d\theta} = -\Big(\frac{\partial g}{\partial u}\Big)^{-1}\frac{\partial g}{\partial\theta}.$$

**Step 3: substitute back.**

$$\frac{dL}{d\theta} = \frac{\partial L}{\partial\theta} - \frac{\partial L}{\partial u}\Big(\frac{\partial g}{\partial u}\Big)^{-1}\frac{\partial g}{\partial\theta}.$$

The gradient is explicit; all that's left is *how* to evaluate the middle product. The order of operations decides the cost.

### Option 1 — direct / tangent-linear (forward)

Build the sensitivity $du/d\theta$ (size $n\times p$) one column at a time. Column $j$ (how the state responds to parameter $\theta_j$) is the linear solve

$$\frac{\partial g}{\partial u}\,\Big(\frac{du}{d\theta}\Big)_{\!\cdot j} = -\Big(\frac{\partial g}{\partial\theta}\Big)_{\!\cdot j}.$$

- Same matrix $\partial g/\partial u$, a different right-hand side for each parameter — so $p$ solves, one per parameter.
- Plug the result into Step 1 to get $dL/d\theta$.
- This is the tangent-linear model: push a perturbation of $\theta$ *forward* through the system. At the AD level, forward mode: a JVP (`jax.jvp`), cheap only when $p$ is small.

### Option 2 — adjoint / cotangent (reverse)

- Evaluate right to left: we never need $du/d\theta$ alone, only its product with $\partial L/\partial u$.
- Group the leading factors into the adjoint variable $\lambda\in\mathbb{R}^n$:

$$\lambda^{\top} := \frac{\partial L}{\partial u}\Big(\frac{\partial g}{\partial u}\Big)^{-1}
\quad\Longleftrightarrow\quad
\Big(\frac{\partial g}{\partial u}\Big)^{\!\top}\lambda = \Big(\frac{\partial L}{\partial u}\Big)^{\!\top}.$$

- That is the **adjoint equation**: one transposed solve, independent of $p$.
- Then $\dfrac{dL}{d\theta} = \dfrac{\partial L}{\partial\theta} - \lambda^{\top}\dfrac{\partial g}{\partial\theta}$ — a few cheap dot products.
- Propagates a cotangent *backward*; at the AD level, reverse mode (a VJP, `jax.vjp`). ($\lambda$ is the constraint's Lagrange multiplier.)

::: {.callout-important}
## The asymmetry that makes ML and inversion possible
A scalar loss with many parameters: the direct method costs $O(p)$ solves, the adjoint costs one — regardless of $p$. That is why we can fit models with millions of parameters.
:::

Same gradient, very different cost. Confirm on a small linear constraint $g(u,\theta)=Au-b(\theta)=0$:

In [2]:
# Constrained problem:  g(u, theta) = A u - b(theta) = 0  ->  u(theta) = A^{-1} b(theta).
# Loss L(u) = 1/2 ||u - u_target||^2.  DIRECT (one solve per parameter) vs ADJOINT (one solve).
n, p = 6, 5                                                     # state dim, number of parameters
Mn = jax.random.normal(jax.random.PRNGKey(0), (n, n))
A  = Mn @ Mn.T + n * jnp.eye(n)                                 # SPD system matrix
B  = jax.random.normal(jax.random.PRNGKey(1), (n, p))          # b(theta) = B @ theta
theta    = jax.random.normal(jax.random.PRNGKey(2), (p,))
u_target = jnp.ones(n)

def solve_u(theta): return jnp.linalg.solve(A, B @ theta)      # the forward solve, u(theta)
u    = solve_u(theta)
dLdu = u - u_target                                            # dL/du

# DIRECT / tangent-linear:  build du/dtheta column by column -> one linear solve PER PARAMETER
dudtheta    = jnp.stack([jnp.linalg.solve(A, B[:, j]) for j in range(p)], axis=1)   # p solves
grad_direct = dLdu @ dudtheta

# ADJOINT / cotangent:  ONE transposed solve, then dot products
lam          = jnp.linalg.solve(A.T, dLdu)                     # (dg/du)^T lambda = (dL/du)^T  <- 1 solve
grad_adjoint = lam @ B                                         # dL/dtheta = -lambda^T (dg/dtheta), dg/dtheta = -B

g_jax = jax.grad(lambda th: 0.5 * jnp.sum((solve_u(th) - u_target) ** 2))(theta)
print(f"parameters p = {p}")
print(f"  direct  : {p} linear solves   grad = {np.round(np.array(grad_direct), 5)}")
print(f"  adjoint : 1 linear solve      grad = {np.round(np.array(grad_adjoint), 5)}")
print(f"  adjoint matches direct: {bool(jnp.allclose(grad_direct, grad_adjoint))}"
      f"   |   matches jax.grad: {bool(jnp.allclose(grad_adjoint, g_jax))}")

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


parameters p = 5
  direct  : 5 linear solves   grad = [ 0.25687 -0.03022  0.0516  -0.32131  0.01129]
  adjoint : 1 linear solve      grad = [ 0.25687 -0.03022  0.0516  -0.32131  0.01129]
  adjoint matches direct: True   |   matches jax.grad: True


## 3 · The same equation everywhere: solvers and neural networks

Nothing above assumed $g$ was a PDE. Any computation that builds $u$ from $\theta$ through a chain of steps defines a constraint, and reverse-mode AD is the adjoint solve for it. The shape of $\partial g/\partial u$ sets the shape of the transposed solve.

### Example A — a time-stepping solver

Stack every time level as an unknown, $U=(u_0,\dots,u_N)$, each $u_k\in\mathbb{R}^d$. One constraint per step:

$$g_0 = u_0 - u_{\text{init}},\qquad g_{k+1} = u_{k+1} - \Phi(u_k,\theta)\quad (k=0,\dots,N-1).$$

Each step couples only to the previous one, so $\partial g/\partial U$ is block lower-bidiagonal ($I$ on the diagonal, $-\Phi'_k\equiv-\partial\Phi(u_k,\theta)/\partial u_k\in\mathbb{R}^{d\times d}$ below); its transpose is upper-bidiagonal:

$$\frac{\partial g}{\partial U}=
\begin{pmatrix}
I & & & \\
-\Phi'_0 & I & & \\
& \ddots & \ddots & \\
& & -\Phi'_{N-1} & I
\end{pmatrix},
\qquad
\Big(\frac{\partial g}{\partial U}\Big)^{\!\top}=
\begin{pmatrix}
I & -\Phi'^{\top}_0 & & \\
& I & \ddots & \\
& & \ddots & -\Phi'^{\top}_{N-1} \\
& & & I
\end{pmatrix}.$$

- An upper-triangular system solves by back-substitution, bottom row up.
- Each row is one *mini adjoint equation*; together they decouple $\big(\partial g/\partial U\big)^{\!\top}\lambda = \partial L/\partial U$ into:

$$\lambda_N = \frac{\partial L}{\partial u_N},\qquad
\lambda_k = \frac{\partial L}{\partial u_k} + \Phi'^{\top}_k\,\lambda_{k+1}\ \ (k=N\!-\!1,\dots,0).$$

The trajectory's adjoint is just the cotangent $\lambda$ carried backward, one transpose-Jacobian per step — possible only because the causal structure makes $\partial g/\partial U$ triangular.

::: {.callout-note}
## Block back-substitution is the scalar algorithm, blockwise
A scalar upper-triangular solve $Ux=b$ runs bottom-up: $x_k = \big(b_k - \sum_{j>k} U_{kj}\,x_j\big)/U_{kk}$. A *block* triangular system uses the identical algorithm — just promote scalars to blocks: "multiply by $U_{kj}$" becomes a matrix–vector product, and "divide by $U_{kk}$" becomes "apply $U_{kk}^{-1}$" — i.e. *solve* the small system $U_{kk}\,x_k=\text{rhs}$, never forming the inverse.

- *Explicit step* — diagonal blocks are $I$, so the divide is trivial: pure substitution, exactly the recurrence above.
- *Implicit step* (e.g. backward-Euler) — the diagonal block is the step's own Jacobian, so each "divide" is one small linear solve.

Either way, it is the discrete adjoint of the lecture's glacier solver.
:::

### Building the big $\lambda$, and how the gradient accumulates

No new machinery — the §2 adjoint equation, with $u$ taken to be the whole stacked state $U$:

$$\Big(\frac{\partial g}{\partial U}\Big)^{\!\top}\lambda = \Big(\frac{\partial L}{\partial U}\Big)^{\!\top}.$$

- $\partial g/\partial U$ is block lower-bidiagonal, so solving it *is* the back-substitution above.
- The backward recurrence is just how we build the big $\lambda=(\lambda_0,\dots,\lambda_N)$, one block per step.

The gradient is the other §2 formula, $\dfrac{dL}{d\theta}=\dfrac{\partial L}{\partial\theta}-\lambda^{\top}\dfrac{\partial g}{\partial\theta}$. With $\partial g/\partial\theta$ stacked over steps (row $k{+}1$ is $-\partial\Phi(u_k,\theta)/\partial\theta$), it becomes a sum over steps:

$$\frac{dL}{d\theta} = \sum_{k=0}^{N-1}\lambda_{k+1}^{\top}\,\frac{\partial\Phi(u_k,\theta)}{\partial\theta}.$$

That sum is the accumulation:

- *Per-step* parameters $\theta_k$ — only one term is nonzero.
- *Shared* parameters (the UDE / NN case, same network every step) — every step contributes and they add. This is why backprop sums a weight's gradient over all the places it appears.

The demo builds $\lambda$ backward, then sums it into the gradient, for a shared $\theta$:

In [3]:
# A few steps of a 2-D NONLINEAR system, ONE shared parameter theta used at EVERY step -- like a UDE.
d, N, dt = 2, 4, 0.4
M = jnp.array([[-1.0, 0.8], [-0.8, -1.0]])
theta = 0.9                                              # SHARED across all steps
u0, target = jnp.array([1.0, 0.5]), jnp.array([0.2, -0.1])

def step(u, th): return u + dt * th * (M @ jnp.tanh(u))  # Phi(u, theta), mild nonlinearity
def rollout(th):
    us = [u0]
    for _ in range(N): us.append(step(us[-1], th))
    return jnp.stack(us)
def loss(th): return jnp.sum((rollout(th)[-1] - target) ** 2)

us = rollout(theta)
# Phi'_k = d step / d u_k = I + dt*theta * M @ diag(1 - tanh(u_k)^2)   (varies per step now)
Pp = [jnp.eye(d) + dt * theta * (M @ jnp.diag(1 - jnp.tanh(us[k]) ** 2)) for k in range(N)]

# (1) the BIG lambda: full transposed solve, and the backward recurrence -- identical
n = (N + 1) * d
G = jnp.eye(n)
for k in range(N): G = G.at[(k+1)*d:(k+2)*d, k*d:(k+1)*d].set(-Pp[k])   # block lower-bidiagonal
dLdU    = jnp.zeros(n).at[N*d:].set(2.0 * (us[-1] - target))            # loss touches only u_N
lam_big = jnp.linalg.solve(G.T, dLdU).reshape(N + 1, d)
lam = [None] * (N + 1); lam[N] = 2.0 * (us[-1] - target)
for k in range(N - 1, -1, -1): lam[k] = Pp[k].T @ lam[k + 1]
print("the big lambda, built BACKWARD (lambda_N -> lambda_0):")
for k in range(N, -1, -1): print(f"   lambda_{k} = {np.round(np.array(lam[k]), 4)}")
print("   full-system solve == backward recurrence:", bool(jnp.allclose(lam_big, jnp.stack(lam))))

# (2) gradient = SUM over steps of lambda_{k+1} . dPhi/dtheta,  with dPhi/dtheta = dt * (M @ tanh(u_k))
print("\ndL/dtheta accumulates one contribution per step (theta is shared):")
run = 0.0
for k in range(N):
    c = float(lam_big[k + 1] @ (dt * (M @ jnp.tanh(us[k]))))
    run += c
    print(f"   + step {k}:  lambda_{k+1} . dPhi/dtheta = {c:+.5f}    (running sum {run:+.5f})")
print(f"   = total {run:+.6f}     jax.grad = {float(jax.grad(loss)(theta)):+.6f}")

the big lambda, built BACKWARD (lambda_N -> lambda_0):
   lambda_4 = [-0.01   -0.3358]
   lambda_3 = [ 0.0757 -0.2246]
   lambda_2 = [ 0.1012 -0.1235]
   lambda_1 = [ 0.1007 -0.0509]
   lambda_0 = [ 0.0917 -0.0137]
   full-system solve == backward recurrence: True

dL/dtheta accumulates one contribution per step (theta is shared):
   + step 0:  lambda_1 . dPhi/dtheta = +0.00601    (running sum +0.00601)
   + step 1:  lambda_2 . dPhi/dtheta = +0.00866    (running sum +0.01467)
   + step 2:  lambda_3 . dPhi/dtheta = +0.00910    (running sum +0.02377)
   + step 3:  lambda_4 . dPhi/dtheta = +0.01103    (running sum +0.03480)


   = total +0.034802     jax.grad = +0.034802


### Example B — a neural network is the same problem

The same construction as Example A, with "time step" → "layer." A feedforward network is a composition

$$x_0 = \text{input},\qquad x_{k+1} = f_k(x_k,\theta_k),\qquad L = \ell(x_n),\qquad x_k\in\mathbb{R}^{d_k},$$

so treat each layer's output as an unknown and each layer as a constraint $g_{k+1} = x_{k+1}-f_k(x_k,\theta_k)=0$.

Each layer depends only on the previous one, so — exactly as in Example A — $\partial g/\partial x$ is block lower-bidiagonal, now with $f'_k\equiv\partial f_k/\partial x_k$:

$$\frac{\partial g}{\partial x}=
\begin{pmatrix}
I & & & \\
-f'_0 & I & & \\
& \ddots & \ddots & \\
& & -f'_{n-1} & I
\end{pmatrix},
\qquad
\Big(\frac{\partial g}{\partial x}\Big)^{\!\top}=
\begin{pmatrix}
I & -f'^{\top}_0 & & \\
& I & \ddots & \\
& & \ddots & -f'^{\top}_{n-1} \\
& & & I
\end{pmatrix}.$$

- Diagonal blocks are identities, so no inverse is ever needed — the transposed solve is pure back-substitution.
- The off-diagonal blocks $f'_k\in\mathbb{R}^{d_{k+1}\times d_k}$ are rectangular when layers change width; the structure stays triangular. (Skip connections, as in ResNets, just add more sub-diagonals — still triangular, still back-substitution.)
- Back-substitution from the bottom is exactly *backpropagation*:

$$\lambda_n = \Big(\frac{\partial\ell}{\partial x_n}\Big)^{\!\top},\qquad
\lambda_k = \Big(\frac{\partial f_k}{\partial x_k}\Big)^{\!\top}\lambda_{k+1},\qquad
\frac{\partial L}{\partial\theta_k} = \Big(\frac{\partial f_k}{\partial\theta_k}\Big)^{\!\top}\lambda_{k+1}.$$

Same three reasons the adjoint is the right tool here as for a glacier: scalar loss, many parameters, triangular $\partial g/\partial x$. For an MLP $f_k(x)=\sigma(W_k x+b_k)$ this is the textbook rule $\lambda_k=W_k^{\!\top}\big(\sigma'(z_k)\odot\lambda_{k+1}\big)$ — matching `jax.grad`:

In [4]:
import jax.random as jr

# A small MLP: tanh hidden layers, linear output.  Params = list of (W, b).
sizes = [4, 16, 16, 1]
keys  = jr.split(jr.PRNGKey(0), len(sizes) - 1)
params = [(0.6 * jr.normal(k, (n_out, n_in)), jnp.zeros(n_out))
          for k, n_in, n_out in zip(keys, sizes[:-1], sizes[1:])]
x_in, y_tgt = jnp.array([0.3, -0.7, 1.1, 0.2]), jnp.array([0.5])

def mlp_loss(params, x):
    a = x
    for W, b in params[:-1]:
        a = jnp.tanh(W @ a + b)          # hidden layers
    W, b = params[-1]
    return jnp.sum((W @ a + b - y_tgt) ** 2)   # linear output + squared error

g_ad = jax.grad(mlp_loss)(params, x_in)        # reverse-mode AD

def hand_backprop(params, x):
    # forward pass, storing the states the backward pass will need
    a, acts, zs = x, [x], []
    for W, b in params[:-1]:
        z = W @ a + b; a = jnp.tanh(z); zs.append(z); acts.append(a)
    W, b = params[-1]; out = W @ a + b
    # backward pass = the adjoint recurrence  lambda_l = (dPhi_l/da_l)^T lambda_{l+1}
    lam = 2.0 * (out - y_tgt)                   # lambda = dL/d(output)
    grads = [(jnp.outer(lam, acts[-1]), lam)]   # output-layer gradients
    lam = params[-1][0].T @ lam                 # push cotangent into last hidden activation
    for l in range(len(params) - 2, -1, -1):
        delta   = (1.0 - jnp.tanh(zs[l]) ** 2) * lam      # through the tanh
        grads.insert(0, (jnp.outer(delta, acts[l]), delta))
        lam = params[l][0].T @ delta                      # (dPhi/da)^T = W^T (.)
    return grads

g_hand = hand_backprop(params, x_in)
worst = max(float(jnp.max(jnp.abs(a - b)))
            for (gw, gb), (hw, hb) in zip(g_ad, g_hand) for a, b in [(gw, hw), (gb, hb)])
print(f"max |AD - hand-coded backprop| over all weights & biases = {worst:.2e}")

max |AD - hand-coded backprop| over all weights & biases = 4.44e-16


::: {.callout-important}
## Punchline
A glacier solver and a neural network look nothing alike, yet their gradients are the same computation: write the model as constraints $g=0$, then solve the single transposed (adjoint) system $(\partial g/\partial u)^{\!\top}\lambda=(\partial L/\partial u)^{\!\top}$. Reverse-mode AD does exactly this, automatically — so "backprop is the adjoint method" is literally true, and a UDE (a network inside a solver) is just one constraint system nested in another.
:::

## 4 · Discrete vs. continuous adjoints

So far, the *discrete* adjoint: differentiate the program as written, so the gradient is exact for the discrete solver (it passes a finite-difference check to machine precision). A second route exists — two ways to apply the same idea:

- *Discrete* (discretize, then differentiate) — the adjoint of the discrete stepper, i.e. the recurrence of §3. What reverse-mode AD produces; exactly consistent with the solver you ran.
- *Continuous* (differentiate, then discretize) — derive an adjoint *differential equation* on paper, then discretize and solve it (often by integrating backward in time). It approximates the discrete gradient: it converges with refinement, but need not match bit-for-bit, and a mismatched discretization can be inconsistent.

Which to use:

- *Continuous* — can be lighter on memory (integrate an adjoint ODE backward instead of storing the trajectory); attractive for long, smooth integrations.
- *Discrete* — exact, robust, and free from AD; the right default, and what the glacier UDE uses.

One caveat for diffusive systems like the SIA: running the dynamics backward in time is ill-posed, so the backward-integration form of the continuous adjoint is unstable — another reason to prefer the discrete adjoint there.

## 5 · Checkpointing: differentiating through long solves

::: {.callout-note}
## Common misconception
By default, neither JAX nor PyTorch checkpoints. The default reverse pass is maximum-memory, zero-recompute: save every intermediate, consume it in reverse.

- *PyTorch (eager):* saves activations; opt-in recompute is `torch.utils.checkpoint`.
- *JAX:* `jax.grad` saves residuals; opt-in rematerialization is `jax.checkpoint` (`jax.remat`).
- *Caveats:* XLA can rematerialize to fit a memory budget (heuristic), and `torch.compile`'s min-cut partitioner makes automatic save-vs-recompute choices — neither is the principled $O(\log N)$ schedule.
:::

Checkpointing trades memory for recomputation: store a few states, recompute the rest during the backward pass. The optimal schedule (Griewank–Walther `revolve`) is $O(\log N)$ memory at $O(N\log N)$ compute.

::: {.callout-important}
## Why this matters for time-dependent inversions and UDEs
Training a UDE means backpropagating through every time step of the forward solve. The adjoint recurrence of §3 needs each forward state $u_k$ to build $(\partial\Phi_k/\partial u_k)^{\top}$, so storing all of them is $O(N)$ in the number of steps:

- our toy glacier: ~600 steps for a spin-up — storing all is fine;
- a real inversion (Bolibar et al.: ~$10^5$ ODEs over a 2-D grid, many years): storing every state is infeasible.

Checkpointing keeps only $O(\log N)$ states and recomputes the rest forward — which is safe because the forward solve only ever runs forward in time. This is why the lecture's solver passes `RecursiveCheckpointAdjoint`.
:::

Hand-rolled `revolve` on a chain of $N$ time steps — to see the memory/compute trade and confirm the gradient is unchanged:

In [6]:
N, theta = 64, 0.99
def phi(y):  return np.tanh(y) + theta * y
def dphi(y): return (1 - np.tanh(y) ** 2) + theta

def store_all(y0):
    ys = [y0]
    for _ in range(N): ys.append(phi(ys[-1]))
    lam = ys[N]
    for k in range(N - 1, -1, -1): lam = dphi(ys[k]) * lam
    return lam, len(ys), N

def checkpointed(y0, s):
    ckpts, y = {0: y0}, y0
    for k in range(1, N + 1):
        y = phi(y)
        if k % s == 0: ckpts[k] = y
    lam, fwd, peak = y, N, max(len(ckpts), s)
    for k in range(N - 1, -1, -1):
        if k in ckpts:
            yk = ckpts[k]
        else:
            base = max(c for c in ckpts if c < k); yk = ckpts[base]
            for _ in range(k - base): yk = phi(yk); fwd += 1
        lam = dphi(yk) * lam
    return lam, peak, fwd

g0, m0, f0 = store_all(1.0)
g1, m1, f1 = checkpointed(1.0, s=8)
print(f"store-all    : ~{m0:2d} states, {f0:3d} forward evals")
print(f"checkpointed : ~{m1:2d} states, {f1:3d} forward evals   (same gradient: {abs(g0-g1) < 1e-9})")

def deep(x):
    for _ in range(50): x = jnp.tanh(x) + 0.1 * x
    return jnp.sum(x ** 2)
x0 = jnp.linspace(-1, 1, 8)
print("jax.grad == jax.grad(jax.checkpoint(...)):", bool(jnp.allclose(jax.grad(deep)(x0), jax.grad(jax.checkpoint(deep))(x0))))

store-all    : ~65 states,  64 forward evals
checkpointed : ~ 9 states, 288 forward evals   (same gradient: True)


jax.grad == jax.grad(jax.checkpoint(...)): True


## 6 · Decision guide

| Situation | Approach |
|---|---|
| New model, many params, scalar loss, moderate memory | reverse-mode AD (discrete adjoint) — the default |
| Neural network inside a solver (a UDE) | reverse-mode AD + checkpointing through the solve |
| Long / stiff integration, memory-bound | checkpointed discrete adjoint (revolve) |
| Long, reversible, non-stiff dynamics | continuous adjoint via backward integration ($O(1)$ memory) |
| Implicit / steady-state / fixed-point solve | implicit function theorem at the converged state — don't unroll iterations |
| FEM with a clean variational form | symbolic form diff + high-level adjoint (Firedrake/FEniCS + pyadjoint) |
| AD-hostile black box in the model | hand-write its VJP (`@custom_vjp` / Julia `rrule`); AD handles the rest |
| Legacy Fortran/C | source-transformation AD (Tapenade, TAF/OpenAD) or a hand adjoint |

::: {.callout-tip}
## One line
Reverse-mode AD is the adjoint method, generated automatically and consistently with your discretization. Reach for an explicit or alternative adjoint mainly for memory at scale, reversible dynamics, implicit solves, FEM forms, or code AD can't penetrate.
:::

### References

- MacAyeal (1993), *Control methods in ice-sheet modeling*, J. Glaciol. 39(131).
- Griewank & Walther (2008), *Evaluating Derivatives* (SIAM); Griewank (1992), `revolve`, Optim. Methods Softw. 1.
- Baydin et al. (2018), *AD in machine learning: a survey*, JMLR 18(153).
- Farrell, Ham, Funke & Rognes (2013), *Automated derivation of the adjoint of high-level FE programs*, SIAM J. Sci. Comput. 35(4); Rathgeber et al. (2016), *Firedrake*, ACM TOMS 43(3); Alnæs et al. (2014), *UFL*, ACM TOMS 40(2).
- Chen et al. (2018), *Neural ODEs*, NeurIPS; Kidger (2021), *On Neural Differential Equations* (thesis; `diffrax`).
- Sapienza et al. (2024), *Differentiable programming for differential equations: a review*, arXiv:2406.09699.

---
*Companion to the UDE glaciology lecture.*